# OpenThoughts Dataset Analysis
Analyzing how many samples don't have answers in the OpenThoughts dataset

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from datasets import load_dataset
import sys
sys.path.append('src')
from phantom_reasoner.dataset_loader import get_openthoughts_dataset, extract_text_between_markers

/home/jcl354/.conda/envs/reasoning/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the original dataset
print("Loading original OpenThoughts dataset...")
original_ds = load_dataset("open-thoughts/OpenThoughts3-1.2M", split="train")
print(f"Original dataset size: {len(original_ds)} samples")

Loading original OpenThoughts dataset...
Original dataset size: 1200000 samples


In [3]:
# Load the filtered dataset (only samples with answers)
print("Loading filtered OpenThoughts dataset...")
filtered_ds = get_openthoughts_dataset(skip_null_answers=True, cache_dir=".cache")
print(f"Filtered dataset size: {len(filtered_ds)} samples")

Loading filtered OpenThoughts dataset...


Map: 100%|█████████████████████████| 1200000/1200000 [24:16<00:00, 824.14 examples/s]
Filtering samples without answers: 100%|█| 1200000/1200000 [17:14<00:00, 1160.28 exam

Filtered dataset size: 302360 samples


In [4]:
# Calculate statistics
samples_without_answers = len(original_ds) - len(filtered_ds)
percentage_without_answers = (samples_without_answers / len(original_ds)) * 100

print(f"\n=== Analysis Results ===")
print(f"Original dataset size: {len(original_ds):,} samples")
print(f"Samples with answers: {len(filtered_ds):,} samples")
print(f"Samples without answers: {samples_without_answers:,} samples")
print(f"Percentage without answers: {percentage_without_answers:.2f}%")


=== Analysis Results ===
Original dataset size: 1,200,000 samples
Samples with answers: 302,360 samples
Samples without answers: 897,640 samples
Percentage without answers: 74.80%


In [9]:
print("\n=== Examining samples without answers ===")
samples_without_answers_count = 0
max_examples_to_show = 5

for i, sample in enumerate(original_ds):
    conversations = sample["conversations"]
    
    # Convert conversations list to a single string (same logic as in dataset_loader)
    conversation_text = ""
    for conv in conversations:
        if isinstance(conv, dict):
            # Handle dict format: {"from": "human", "value": "..."}
            role = conv.get("from", "unknown")
            value = conv.get("value", "")
            conversation_text += f"{role}: {value}\n"
        elif isinstance(conv, str):
            # Handle string format directly
            conversation_text += conv + "\n"
    
    answer = extract_text_between_markers(conversation_text, "**Final Answer**", "</think>")
    
    if not answer:
        samples_without_answers_count += 1
        if samples_without_answers_count <= max_examples_to_show:
            print(f"\n--- Sample {i} (no answer found) ---")
            print(f"Contains '**Final Answer**': {'**Final Answer**' in conversation_text}")
            print(f"Contains '</think>': {'</think>' in conversation_text}")
            print(f"Conversation preview: {conversation_text}...")
    
    if samples_without_answers_count >= max_examples_to_show:
        break

print(f"\nFound {samples_without_answers_count} samples without answers in the first {i+1} samples")


=== Examining samples without answers ===

--- Sample 0 (no answer found) ---
Contains '**Final Answer**': False
Contains '</think>': True
Conversation preview: human: I am in desperate need of some ear defenders, so I can program in peace. Unfortunately, I don't have ear defenders. But I do have a pair of headphones, a microphone and a microcontroller, so I thought I'd make some noise-cancelling headphones.
However, there is one tiny problem. I can't program in noisy environments! So you'll need to write the code for me.
Task
Given a constant noisy input, output the input's "complement"; a value that will completely cancel out the input value (0x00 cancels 0xff, 0x7f cancels 0x80). Because the headphones block some noise, also take a value as input that you will divide the "complement" by, using the following algorithm: if (c < 0x80) ceil ( 0x7f - ( (0x7f - c) / n ) ) else floor ( ( (c - 0x80) / n ) + 0x80 ) where c is the "complement" and n is the value to divide by. (If the value e

In [10]:
for i, sample in enumerate(filtered_ds):
    print("prompt: ",sample["prompt"][0]["content"])
    print("answer: ",sample["answer"])
    print("prompt_method: ",sample["prompt_method"])
    print("difficulty: ",sample["difficulty"])
    print("response: ",sample["response"])
    if i >= 5:
        break  # show only 5 full conversations

prompt:  your task is to write a program that uses the Lucas-Lehmer primality test to check whether an input number is prime or not. This prime number candidate has the form 2p-1 with p being prime. (Mersenne prime number)
The Lucas-Lehmer-test
The n-th Mersenne number is prime if it is a divisor of the (n-1)-th Lucas-Lehmer number (LLN).
You can calculate the LLN, similiar to the Fibonacci numbers, based on a recursive algorithm:
L<n> = (L<n-1>)^2 - 2

The LLN for 2 is 14. So, for p=3, LLN<3-1>=14
Evaluation

You have to submit a function that takes p as input (Remember: 2^p-1)
The shortest solution will win.
Your solution has to use the Lucas-Lehmer test
The code must be executable.
Output: boolean value or 1=prime/0=not prime

Tests:
p --> 2^p-1  --> LLN<n-1> --> ?
3 --> 7      --> 14  --> prime
5 --> 31     --> 37634 > prime (1214*31)

Good Luck
answer:  ['\\boxed{True}\n\nBut wait, the problem requires submission of the code as the solution, not the answer to the tests. Wait, no.\

In [ ]:
# Calculate token statistics for responses (OPTIMIZED VERSION)
print("=== Response Token Statistics ===")

# Extract all responses at once and convert to pandas Series for vectorized operations
responses = [sample["response"] for sample in filtered_ds]
response_series = pd.Series(responses)

# Vectorized word counting - much faster than loop
response_lengths = response_series.str.split().str.len().fillna(0).values

# Calculate summary statistics using numpy (already fast)
print(f"Total samples with responses: {len(response_lengths)}")
print(f"Mean response length (words): {np.mean(response_lengths):.2f}")
print(f"Median response length (words): {np.median(response_lengths):.2f}")
print(f"Standard deviation: {np.std(response_lengths):.2f}")
print(f"Minimum response length: {np.min(response_lengths)}")
print(f"Maximum response length: {np.max(response_lengths)}")

# Percentiles
percentiles = [10, 25, 50, 75, 90, 95, 99]
print(f"\nPercentiles:")
for p in percentiles:
    value = np.percentile(response_lengths, p)
    print(f"  {p}th percentile: {value:.0f} words")

# Distribution analysis (vectorized)
print(f"\n=== Distribution Analysis ===")
bins = [0, 100, 500, 1000, 2000, 3000, 4000]
labels = ['0-100', '101-500', '501-1000', '1001-2000', '2001-3000', '3001-4000']
hist, _ = np.histogram(response_lengths, bins=bins)
total = len(response_lengths)

for i, (label, count) in enumerate(zip(labels, hist)):
    percentage = (count / total) * 100
    print(f"Responses with {label} words: {count:,} ({percentage:.1f}%)")

# Sample response lengths (vectorized)
print(f"\n=== Sample Response Lengths ===")
print("First 20 response lengths (words):")
for i, length in enumerate(response_lengths[:20]):
    print(f"  Sample {i+1}: {length} words")
